# RNA3D Pilot10 — trRosettaRNA2 structural evaluation and four-model comparison

**CPU only.** This evaluates the **completed** trRosettaRNA2 single-sequence A3M run (`trrosettarna2_pilot10_single_20261114_v1`) against the same ten frozen experimental natives and residue maps used for NuFold, DRfold2, and RhoFold+.

- **No inference, no GPU, no GitHub write.** Local read-only pinned helper checkout and US-align source are cloned to Colab `/content`.
- **Mapping/scoring:** keep experimentally observed target positions, use `USalign mapped_prediction native_eval -mol RNA -TMscore 1`, record **TM-score normalized by native Structure_2**, aligned length and RMSD.
- **Four-model comparison:** reuse **archived** previous three-model scores; do not silently rescore or modify them.
- **Strict gates:** prediction hash and sequence, correct model commit/input mode, mapping/natives, prior scored table, US-align parser self-test, all ten successful scores; save to a *new* immutable Drive folder with SHA-256 manifest.
- **Scope:** exploratory 10-target results. Different inference preprocessing, missing experimental residues, and unresolved training overlap prevent definitive ranking or template-scarcity claims.

**Run each cell in order on Google Colab CPU.** On error, do **not** overwrite existing Drive run directories; send the traceback.

## 1 — Mount Google Drive (same account as `RNA3D_Benchmark`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 — Frozen run paths and preflight

In [ ]:
from pathlib import Path
import csv, hashlib, json, math, os, re, subprocess, sys, tempfile, shutil
from datetime import datetime, timezone
from statistics import mean

ROOT=Path('/content/drive/MyDrive/RNA3D_Benchmark')
TR_RUN=ROOT/'trRosettaRNA2/runs/trrosettarna2_pilot10_single_20261114_v1'
NU_RUN=ROOT/'NuFold/runs/pilot10_single_20261018T203832Z'
NATIVE=ROOT/'native/pilot_v1/native_eval'
RHO_EVAL=ROOT/'RhoFold/runs/rhofoldplus_pilot10_single_relax1000_20261111_v1/evaluations/rhofoldplus_threeway_usalign20260527_20261112_v1'
TARGETS=['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B','9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
MODEL_COMMIT='d40a40e36617cb4004a11b0a7603167415ed9c52'
PROJECT_COMMIT='c79a78b0a35be00a31aadeb04a6ebec5f2a0004f'
USALIGN_TAG='20260527'
RUN_ID='trrosettarna2_pilot10_fourmodel_eval_20261115_v1'
OUT=ROOT/'Evaluations'/RUN_ID
STAGE=Path('/content/rna3d_trrosetta_four_model_stage_v1')
PROJECT=Path('/content/rna3d_tr_eval_helpers')
US_SRC=Path('/content/USalign_tr_eval')
US_BIN=US_SRC/'USalign'

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def read_csv(p):
    with Path(p).open(newline='') as f:return list(csv.DictReader(f))

assert len(TARGETS)==len(set(TARGETS))==10
for p in [ROOT,TR_RUN,NU_RUN,NATIVE,RHO_EVAL]:
    assert p.is_dir(),f'Missing Drive folder: {p}'
assert not OUT.exists(),f'Output already exists: {OUT}; never overwrite previous runs.'
assert not STAGE.exists(),f'Staging folder already exists: {STAGE}; inspect before reuse.'
print('Predictions:',TR_RUN)
print('Reference:',NATIVE)
print('Archived 3-model evaluation:',RHO_EVAL)
print('NEW output:',OUT)

## 3 — Pinned evaluation helpers and US-align binary

In [ ]:
def run(cmd,cwd=None):
    print('$',' '.join(map(str,cmd)))
    subprocess.run(list(map(str,cmd)),cwd=cwd,check=True)

if not PROJECT.exists():
    run(['git','clone','--filter=blob:none','https://github.com/ABDULSALAM589/rna3d-template-sparse-benchmark.git',PROJECT])
run(['git','-C',PROJECT,'checkout','--detach',PROJECT_COMMIT])
assert subprocess.check_output(['git','-C',PROJECT,'rev-parse','HEAD'],text=True).strip()==PROJECT_COMMIT
sys.path.insert(0,str(PROJECT/'scripts/nufold'))
from common import pdb_residues,validate_ranked_pdb,read_native_positions,write_mapped_prediction

if not US_SRC.exists():
    run(['git','clone','--depth','1','--branch',USALIGN_TAG,'https://github.com/pylelab/USalign.git',US_SRC])
head=subprocess.check_output(['git','-C',US_SRC,'rev-parse','HEAD'],text=True).strip()
tag=subprocess.check_output(['git','-C',US_SRC,'rev-parse',f'refs/tags/{USALIGN_TAG}^{{commit}}'],text=True).strip()
assert head==tag,'Wrong US-align tag'
if not US_BIN.is_file(): run(['make'],cwd=US_SRC)
assert US_BIN.is_file() and os.access(US_BIN,os.X_OK)
print('Mapping helper commit:',PROJECT_COMMIT)
print('US-align source:',head,'executable SHA-256:',sha256(US_BIN))

## 4 — Verify completed run, 10 PDBs, sequences, maps, archived 3-model scores

In [ ]:
status_path=TR_RUN/'run_status.json'
assert status_path.is_file(),status_path
status=json.loads(status_path.read_text())
assert status.get('status')=='complete' and status.get('passed')==status.get('targets')==10
assert status.get('model_commit')==MODEL_COMMIT
assert status.get('mode')=='single-sequence A3M' and status.get('native_used_for_inference') is False

rows=read_csv(TR_RUN/'batch_summary.csv')
assert len(rows)==10 and set(r['target_id'] for r in rows)==set(TARGETS)
by_id={r['target_id']:r for r in rows}

# Reference model scores: archive is immutable and verified by its FINALIZED marker.
prior_final=json.loads((RHO_EVAL/'FINALIZED.json').read_text())
assert prior_final['status']=='complete'
for filename,digest in prior_final['files'].items():
    assert sha256(RHO_EVAL/filename)==digest, f'Prior evaluation file changed: {filename}'
paired=read_csv(RHO_EVAL/'three_model_paired.csv')
assert len(paired)==10 and set(x['target_id'] for x in paired)==set(TARGETS)
prior={r['target_id']:r for r in paired}

inputs=[]
for t in TARGETS:
    r=by_id[t]
    assert r['model']=='trRosettaRNA2' and r['model_commit']==MODEL_COMMIT
    assert r['input_mode']=='single_sequence_a3m' and int(r['msa_depth'])==1
    assert r['status']=='PASS' and int(r['exit_code'])==0
    fas=TR_RUN/t/f'{t}.fasta'
    a3m=TR_RUN/t/f'{t}.a3m'
    pred=TR_RUN/t/'model_1_relaxed200.pdb'
    native=NATIVE/f'{t}_native_eval.pdb'
    rmap=NATIVE/f'{t}_residue_map.csv'
    assert all(p.is_file() for p in [fas,a3m,pred,native,rmap]),f'{t}: missing file'
    seq=''.join(l.strip() for l in fas.read_text().splitlines() if not l.startswith('>')).upper()
    assert seq and set(seq)<=set('ACGU') and len(seq)==int(r['sequence_length'])
    assert sha256(a3m)==r['input_a3m_sha256'],f'{t}: A3M hash mismatch'
    assert sha256(pred)==r['output_pdb_sha256'],f'{t}: PDB hash mismatch'
    assert int(r['output_residues'])==len(seq)
    pi=validate_ranked_pdb(pred,len(seq))
    assert pi['sequence']==seq,f'{t}: predicted sequence differs from FASTA'
    positions=read_native_positions(rmap)
    assert positions and positions==sorted(set(positions)) and 1<=positions[0]<=positions[-1]<=len(seq)
    native_rs=pdb_residues(native)
    assert len(native_rs)==len(positions),f'{t}: native residue count mismatch'
    assert ''.join(x['one'] for x in native_rs)==''.join(seq[i-1] for i in positions),f'{t}: native sequence mismatch'
    assert int(prior[t]['native_residues'])==len(positions)
    assert int(prior[t]['input_length'])==len(seq)
    inputs.append(dict(target_id=t,seq=seq,pred=pred,native=native,map=rmap,positions=positions,
                       length=len(seq),native_count=len(positions),runtime=float(r['runtime_seconds']),
                       output_sha=r['output_pdb_sha256']))
print('INPUT PREFLIGHT PASSED: 10/10 predictions + native residue maps + archived 3-model scores.')
for q in inputs: print(q['target_id'],q['length'],'nt, native observed',q['native_count'])

## 5 — US-align parser self-test and one-target dry run

In [ ]:
def parse_usalign(stdout,n):
    m=re.search(r'Aligned length\s*=\s*(\d+)\s*,\s*RMSD\s*=\s*([0-9.eE+-]+)',stdout,re.I)
    if not m:raise ValueError('Missing aligned length and RMSD; raw output follows:\n'+stdout[-1500:])
    vals=re.findall(r'TM-score\s*=\s*([0-9.eE+-]+)\s*\(normalized by length of Structure_2\b[^)]*\)',stdout,re.I)
    if not vals:raise ValueError('Missing native (Structure_2)-normalized TM-score; output:\n'+stdout[-1500:])
    aligned=int(m.group(1));rmsd=float(m.group(2));tm=float(vals[-1])
    assert 0<aligned<=n and 0<=tm<=1.00001 and math.isfinite(tm) and math.isfinite(rmsd) and rmsd>=0
    return dict(tm_score=tm,rmsd_angstrom=rmsd,aligned_residues=aligned,native_coverage=aligned/n)

mock='''Aligned length= 13, RMSD=  1.93, Seq_ID=n_identical/n_aligned=1.000
TM-score= 0.19096 (normalized by length of Structure_1: L=13, d0=0.40)
TM-score= 0.19096 (normalized by length of Structure_2: L=13, d0=0.40)'''
assert parse_usalign(mock,13)==dict(tm_score=0.19096,rmsd_angstrom=1.93,aligned_residues=13,native_coverage=1.0)
print('Parser self-test PASS')

STAGE.mkdir(parents=True)
(STAGE/'mapped').mkdir()
(STAGE/'logs').mkdir()

def score(q):
    t=q['target_id']
    mapped=STAGE/'mapped'/f'{t}_trrosettarna2_mapped.pdb'
    n=write_mapped_prediction(q['pred'],mapped,q['seq'],q['positions'])
    assert n==q['native_count']
    p=subprocess.run([str(US_BIN),str(mapped),str(q['native']),'-mol','RNA','-TMscore','1'],
                     capture_output=True,text=True)
    log=p.stdout+('\n--- STDERR ---\n'+p.stderr if p.stderr else '')
    (STAGE/'logs'/f'{t}_usalign.txt').write_text(log)
    if p.returncode:raise RuntimeError(f'{t}: US-align exited {p.returncode}: {log[-1500:]}')
    parsed=parse_usalign(p.stdout,q['native_count'])
    assert parsed['aligned_residues']==q['native_count'],f'{t}: incomplete identity mapping'
    return dict(target_id=t,input_length=q['length'],native_residues=q['native_count'],
                native_fraction=q['native_count']/q['length'],model='trRosettaRNA2',
                input_mode='single_sequence_a3m',prediction_sha256=q['output_sha'],
                mapped_sha256=sha256(mapped),runtime_seconds=q['runtime'],**parsed)

first=score(inputs[0]);print('DRY RUN:',first)

## 6 — Evaluate all ten predictions

In [ ]:
scores=[first]
for q in inputs[1:]:
    row=score(q)
    scores.append(row)
    print(q['target_id'],'TM=',f"{row['tm_score']:.5f}",'RMSD=',f"{row['rmsd_angstrom']:.2f}")
assert len(scores)==10 and {x['target_id'] for x in scores}==set(TARGETS)
print('trRosettaRNA2 COMPLETED: 10/10')

## 7 — Four-model paired comparison (archived previous three scores)

In [ ]:
# Strictly verify the paired CSV schema instead of silently substituting columns.
print('Archived paired columns:',list(paired[0].keys()))

# The pre-existing 3-model paired CSV uses these canonical TM/RMSD names.
# Allow explicitly recognized historical naming variants only.
def choose(row,choices):
    for name in choices:
        if name in row and str(row[name]).strip()!='':return float(row[name])
    raise KeyError(f'Expected one of {choices}; actual columns: {list(row)}')

names={
 'nufold_tm':['nufold_rank1_tm_score','nufold_tm_score','nufold_tm','nufold_tm_native','nufold_tm_score_native'],
 'drfold2_tm':['drfold2_tm_score','drfold2_tm','drfold2_tm_native'],
 'rhofoldplus_tm':['rhofoldplus_single_tm_score','rhofoldplus_tm_score','rhofoldplus_tm','rhofold_tm_score','rhofold_tm'],
 'nufold_rmsd':['nufold_rank1_rmsd_angstrom','nufold_rmsd_angstrom','nufold_rmsd'],
 'drfold2_rmsd':['drfold2_rmsd_angstrom','drfold2_rmsd'],
 'rhofoldplus_rmsd':['rhofoldplus_single_rmsd_angstrom','rhofoldplus_rmsd_angstrom','rhofoldplus_rmsd','rhofold_rmsd_angstrom']}

four=[]
for row in scores:
    t=row['target_id']; p=prior[t]
    prev={key:choose(p,choices) for key,choices in names.items()}
    four.append(dict(target_id=t,input_length=row['input_length'],native_residues=row['native_residues'],
                     native_fraction=row['native_fraction'],**prev,
                     trrosettarna2_tm=row['tm_score'],trrosettarna2_rmsd=row['rmsd_angstrom'],
                     trrosettarna2_native_coverage=row['native_coverage']))

avg={m:{'mean_tm':mean(r[f'{m}_tm'] for r in four),
        'mean_rmsd':mean(r[f'{m}_rmsd'] for r in four)}
     for m in ['nufold','drfold2','rhofoldplus','trrosettarna2']}
subset=[r for r in four if r['native_fraction']>=0.8]
avg80={m:{'mean_tm':mean(r[f'{m}_tm'] for r in subset),
          'mean_rmsd':mean(r[f'{m}_rmsd'] for r in subset)}
       for m in avg}
print('ALL 10:',json.dumps(avg,indent=2))
print('NATIVE >=80%:',len(subset),json.dumps(avg80,indent=2))

## 8 — Save new immutable Drive evaluation and SHA-256 FINALIZED marker

In [ ]:
def write_csv(p,rows):
    assert rows
    with Path(p).open('w',newline='') as h:
        w=csv.DictWriter(h,fieldnames=list(rows[0]),lineterminator='\n');w.writeheader();w.writerows(rows)

write_csv(STAGE/'trrosettarna2_scores.csv',scores)
write_csv(STAGE/'four_model_paired.csv',four)
summary={
 'status':'complete','run_id':RUN_ID,'created_utc':datetime.now(timezone.utc).isoformat(),
 'model':'trRosettaRNA2','model_commit':MODEL_COMMIT,'input_mode':'single_sequence_a3m',
 'native_used_for_inference':False,'targets':len(TARGETS),'model_scores':avg,
 'native_fraction_ge_0p8_targets':len(subset),'model_scores_native_fraction_ge_0p8':avg80,
 'source_tr_run':'trrosettarna2_pilot10_single_20261114_v1',
 'source_threeway_eval':'rhofoldplus_threeway_usalign20260527_20261112_v1',
 'source_threeway_finalized_sha256':sha256(RHO_EVAL/'FINALIZED.json'),
 'mapping_git_commit':PROJECT_COMMIT,'usalign_tag':USALIGN_TAG,'usalign_source_commit':head,
 'usalign_binary_sha256':sha256(US_BIN),
 'method':'USalign mapped_prediction native_eval -mol RNA -TMscore 1; normalize by Structure_2 native',
 'limitations':['10 target exploratory sample','models have differing inference input conditions',
 'some native experimental residues missing','training overlap and template scarcity not established',
 'native-observed-position scoring; short-RNA TM-score may be hard to interpret']}
(STAGE/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
assert not OUT.exists(),'Existing output must never be overwritten'
# Ensure all output files exist locally BEFORE creating the Drive output directory.
files=[STAGE/'trrosettarna2_scores.csv',STAGE/'four_model_paired.csv',STAGE/'summary.json']
files += sorted((STAGE/'mapped').glob('*.pdb'))+sorted((STAGE/'logs').glob('*.txt'))
assert len(list((STAGE/'mapped').glob('*.pdb')))==10
assert len(list((STAGE/'logs').glob('*.txt')))==10
OUT.mkdir(parents=True,exist_ok=False)
try:
    for p in files:
        dst=OUT/p.relative_to(STAGE)
        dst.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(p,dst)
        assert sha256(p)==sha256(dst)
    hashes={str(p.relative_to(STAGE)):sha256(p) for p in files}
    marker={'status':'complete','run_id':RUN_ID,'files':hashes,'created_utc':datetime.now(timezone.utc).isoformat()}
    (OUT/'FINALIZED.json').write_text(json.dumps(marker,indent=2)+'\n')
except Exception:
    print('SAVE INCOMPLETE; inspect new output folder before rerun; do not overwrite it.')
    raise
print('SAVED AND HASH-VERIFIED:',OUT)
print('10/10 SCORED; NO GITHUB WRITES')

## 9 — Read-only post-save verification

In [ ]:
marker=json.loads((OUT/'FINALIZED.json').read_text())
assert marker['status']=='complete' and marker['run_id']==RUN_ID
for rel,digest in marker['files'].items():
    assert sha256(OUT/rel)==digest,rel
assert len(read_csv(OUT/'trrosettarna2_scores.csv'))==10
assert len(read_csv(OUT/'four_model_paired.csv'))==10
print('FINAL VERIFICATION PASS: 10/10 four-model evaluation completed')
print('Results:',OUT)